# Notebook 02 — Feature Engineering

**Goal:** Compute all behavioural features from the enriched scrobble data and produce a user-level feature matrix for clustering.

**Inputs** (from `data/processed/`, all CSV):
- `scrobbles_updated.csv`
- `profiles.csv`
- `artist_genres.csv`
- `audio_features.csv`

**Outputs:**
- `data/processed/user_features.csv` — one row per user, all features

**Features computed:**

| Group | Features |
|---|---|
| Artist diversity | unique_artists, artist_entropy, artist_concentration_20 |
| Genre diversity | unique_genres, genre_entropy, genre_concentration_5, avg_genre_tags_per_play |
| Engagement | total_scrobbles, unique_tracks, track_replay_rate, avg_tracks_per_session, session_count |
| Discovery | discovery_velocity_30d/90d, novelty_ratio, top_artist_play_share |
| Temporal | temporal_hour/dow/month_entropy, morning/evening/weekend_ratio, temporal_stability_pc1..5 |
| Audio profile | mean danceability, energy, valence, tempo, acousticness, instrumentalness |

In [ ]:
import sys
import logging
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import pandas as pd
from src.data.loader import load_artist_genres_csv

# With ~10,000 sampled users the CSV files are small enough to load directly.
scrobbles     = pd.read_csv('../data/processed/scrobbles_updated.csv',
                             parse_dates=['timestamp'])
profiles      = pd.read_csv('../data/processed/profiles.csv')
# load_artist_genres_csv deserialises the JSON-encoded 'genres' column to list[str]
artist_genres = load_artist_genres_csv('../data/processed/artist_genres.csv')
audio_features = pd.read_csv('../data/processed/audio_features.csv')

print(f'Scrobbles     : {len(scrobbles):,} rows, {scrobbles["userid"].nunique():,} users')
print(f'Artist genres : {len(artist_genres):,} artists, '
      f'{artist_genres["spotify_artist_id"].notna().sum():,} matched')
print(f'Audio features: {len(audio_features):,} tracks')

## 1. Artist Diversity Features

In [ ]:
from src.features.diversity import compute_artist_diversity

artist_div = compute_artist_diversity(scrobbles, top_n=20)
print(f'Artist diversity features: {artist_div.shape}')
artist_div.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
artist_div['unique_artists'].hist(ax=axes[0], bins=30)
axes[0].set_title('Unique Artists per User')

artist_div['artist_entropy'].hist(ax=axes[1], bins=30)
axes[1].set_title('Artist Shannon Entropy')

artist_div['artist_concentration_20'].hist(ax=axes[2], bins=30)
axes[2].set_title('Top-20 Artist Concentration')

plt.tight_layout()
plt.show()

## 2. Genre Diversity Features

In [ ]:
from src.features.diversity import compute_genre_diversity

genre_div = compute_genre_diversity(scrobbles, artist_genres, top_n=5)
print(f'Genre diversity features: {genre_div.shape}')
genre_div.describe()

## 3. Temporal Features

Includes PCA-compressed hourly listening profiles to capture **listening pattern stability** while avoiding multicollinearity across 24 hour-of-day columns.

In [ ]:
from src.features.temporal import compute_temporal_features

temporal = compute_temporal_features(scrobbles, pca_components=5)
print(f'Temporal features: {temporal.shape}')
temporal.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
temporal['temporal_hour_entropy'].hist(ax=axes[0], bins=20)
axes[0].set_title('Hour-of-Day Entropy')

temporal['weekend_ratio'].hist(ax=axes[1], bins=20)
axes[1].set_title('Weekend Listen Ratio')

temporal['avg_daily_plays'].hist(ax=axes[2], bins=30)
axes[2].set_title('Avg Daily Plays')

plt.tight_layout()
plt.show()

## 4. Engagement & Discovery Features

In [ ]:
from src.features.engagement import compute_engagement_features

engagement = compute_engagement_features(scrobbles, session_gap_minutes=30)
print(f'Engagement features: {engagement.shape}')
engagement.describe()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(['track_replay_rate', 'avg_tracks_per_session',
                          'discovery_velocity_30d', 'novelty_ratio',
                          'top_artist_play_share', 'session_count']):
    engagement[col].hist(ax=axes[i], bins=25)
    axes[i].set_title(col.replace('_', ' ').title())

plt.tight_layout()
plt.show()

## 5. Audio Feature Profile

In [ ]:
from src.features.engagement import compute_audio_feature_profile

audio_profile = compute_audio_feature_profile(scrobbles, audio_features)
print(f'Audio feature profile: {audio_profile.shape}')
audio_profile.head()

## 6. Build Final Feature Matrix

In [ ]:
import os

# Join all feature blocks (already computed above — no need to reload scrobbles).
feature_matrix = (
    artist_div
    .join(genre_div,     how='outer')
    .join(temporal,      how='outer')
    .join(engagement,    how='outer')
    .join(audio_profile, how='outer')
)

# Add demographics (descriptive only, not used as clustering inputs)
profiles_indexed = profiles.set_index('userid')[['gender', 'age', 'country']]
feature_matrix = feature_matrix.join(profiles_indexed, how='left').sort_index()

os.makedirs('../data/processed', exist_ok=True)
feature_matrix.to_csv('../data/processed/user_features.csv')

print(f'Feature matrix: {feature_matrix.shape[0]} users × {feature_matrix.shape[1]} features')
feature_matrix.head()

## 6b. EDA — Feature Matrix Descriptive Statistics

Inspect the shape, completeness, and distributions of every feature group before feeding the matrix into the clustering pipeline.

In [ ]:
# ── Shape & completeness ────────────────────────────────────────────
print(f'Feature matrix: {feature_matrix.shape[0]:,} users × {feature_matrix.shape[1]} columns')
print(f'Index name: {feature_matrix.index.name}')
print()

missing = feature_matrix.isnull().sum()
missing_pct = (missing / len(feature_matrix) * 100).round(2)
completeness = (
    missing_pct
    .rename('missing_%')
    .to_frame()
    .query('`missing_%` > 0')
    .sort_values('missing_%', ascending=False)
)
if completeness.empty:
    print('No missing values — feature matrix is complete.')
else:
    print('Columns with missing values:')
    print(completeness.to_string())


In [ ]:
# ── Descriptive statistics by feature group ─────────────────────────

feature_groups = {
    'Diversity': [
        'unique_artists', 'artist_entropy', 'artist_concentration_20',
        'unique_genres', 'genre_entropy', 'genre_concentration_5',
        'avg_genre_tags_per_play',
    ],
    'Temporal': [
        'temporal_hour_entropy', 'temporal_dow_entropy', 'temporal_month_entropy',
        'morning_ratio', 'evening_ratio', 'weekend_ratio',
        'listening_days', 'avg_daily_plays',
    ],
    'Engagement': [
        'total_scrobbles', 'unique_tracks', 'track_replay_rate',
        'avg_tracks_per_session', 'session_count', 'avg_session_length_min',
        'discovery_velocity_30d', 'discovery_velocity_90d',
        'novelty_ratio', 'top_artist_play_share',
    ],
    'Audio (mean)': [
        'mean_danceability', 'mean_energy', 'mean_valence', 'mean_tempo',
        'mean_acousticness', 'mean_instrumentalness',
        'mean_liveness', 'mean_speechiness',
    ],
}

for group_name, cols in feature_groups.items():
    present = [c for c in cols if c in feature_matrix.columns]
    if not present:
        continue
    print(f'\n{'='*64}')
    print(f'  {group_name.upper()} FEATURES  ({len(present)} columns)')
    print(f'{'='*64}')
    stats = (
        feature_matrix[present]
        .describe(percentiles=[.05, .25, .50, .75, .95])
        .round(4)
    )
    print(stats.to_string())


In [ ]:
# ── Skewness & outlier summary ──────────────────────────────────────
import numpy as np

numeric = feature_matrix.select_dtypes(include=[np.number])

skew_df = numeric.skew().rename('skewness').to_frame()
skew_df['kurt'] = numeric.kurt()

# Flag columns where |skew| > 2 (may benefit from log transform)
skew_df['highly_skewed'] = skew_df['skewness'].abs() > 2

# IQR-based outlier rate per column
q1 = numeric.quantile(0.25)
q3 = numeric.quantile(0.75)
iqr = q3 - q1
outlier_rate = (
    ((numeric < (q1 - 1.5 * iqr)) | (numeric > (q3 + 1.5 * iqr)))
    .mean()
    .rename('outlier_rate_%')
    * 100
).round(2)

summary = skew_df.join(outlier_rate).sort_values('skewness', ascending=False)
print('Skewness, kurtosis, and IQR-outlier rate per feature:')
print(summary.round(3).to_string())
print(f'\nHighly skewed columns (|skew| > 2): {skew_df["highly_skewed"].sum()}')


In [ ]:
# ── Distribution plots — one panel per feature group ────────────────
import matplotlib.pyplot as plt

for group_name, cols in feature_groups.items():
    present = [c for c in cols if c in feature_matrix.columns]
    if not present:
        continue
    n = len(present)
    ncols = 4
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.5 * nrows))
    axes = axes.flatten() if n > 1 else [axes]
    fig.suptitle(f'{group_name} Features — Distributions', fontsize=14, y=1.01)

    for ax, col in zip(axes, present):
        data = feature_matrix[col].dropna()
        ax.hist(data, bins=40, color='steelblue', edgecolor='white', linewidth=0.4)
        ax.axvline(data.median(), color='red', linewidth=1.2, linestyle='--',
                   label=f'median={data.median():.3f}')
        ax.set_title(col, fontsize=9)
        ax.legend(fontsize=7)
        ax.tick_params(labelsize=7)

    # Hide unused axes
    for ax in axes[len(present):]:
        ax.set_visible(False)

    plt.tight_layout()
    plt.show()


In [ ]:
# ── Pairwise correlations within each feature group ─────────────────
import matplotlib.pyplot as plt
import numpy as np

for group_name, cols in feature_groups.items():
    present = [c for c in cols if c in feature_matrix.columns]
    if len(present) < 2:
        continue
    corr = feature_matrix[present].corr()

    fig, ax = plt.subplots(figsize=(max(6, len(present)), max(5, len(present) - 1)))
    im = ax.imshow(corr.values, cmap='RdBu', vmin=-1, vmax=1)
    ax.set_xticks(range(len(present)))
    ax.set_yticks(range(len(present)))
    ax.set_xticklabels(present, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(present, fontsize=8)
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(f'{group_name} — Correlation Matrix', fontsize=12)

    # Annotate cells
    for i in range(len(present)):
        for j in range(len(present)):
            ax.text(j, i, f'{corr.values[i, j]:.2f}',
                    ha='center', va='center', fontsize=7,
                    color='white' if abs(corr.values[i, j]) > 0.6 else 'black')
    plt.tight_layout()
    plt.show()

    # Flag high correlations
    high = (
        corr.abs()
        .where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .loc[lambda s: s > 0.80]
        .sort_values(ascending=False)
    )
    if not high.empty:
        print(f'{group_name} — pairs with |r| > 0.80:')
        print(high.round(3).to_string())
        print()


## 7. Correlation Analysis & Multicollinearity Check

In [ ]:
# Select numeric clustering features (exclude demographics)
clustering_features = feature_matrix.select_dtypes(include=[np.number]).drop(
    columns=['age'], errors='ignore'
)

corr = clustering_features.corr()

plt.figure(figsize=(20, 16))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, cmap='RdBu', center=0, vmin=-1, vmax=1,
    annot=False, fmt='.2f', linewidths=0.3,
)
plt.title('Feature Correlation Matrix (Clustering Features)')
plt.tight_layout()
plt.savefig('../outputs/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Pairs with |r| > 0.75
high_corr = (
    corr.abs().where(np.tril(np.ones(corr.shape), k=-1).astype(bool))
    .stack().reset_index()
    .rename(columns={0: 'correlation', 'level_0': 'feat_a', 'level_1': 'feat_b'})
    .query('correlation > 0.75')
    .sort_values('correlation', ascending=False)
)
print(f'Highly correlated pairs (|r|>0.75): {len(high_corr)}')
print(high_corr.to_string())

> Note: Remaining multicollinearity is handled at the clustering stage via PCA dimensionality reduction before fitting KMeans/HDBSCAN.

Proceed to **Notebook 03** for clustering.